CEP: Order-Book Linear Robust Model

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../../')

In [2]:
import pandas as pd
from statsmodels.formula import api as smf
from data_helpers.data_loaders import PandasDataLoader 
from data_helpers.data_prep import dynamic_treatments
import numpy as np
from tqdm.auto import tqdm


## Load and filter dataset

In [3]:
model_name = 'ob_rlm'
sample_splits_df = pd.read_feather('../../../data/preprocessed/train_test_split.ft')
time_aggregated_dataset = pd.read_feather('../../../data/preprocessed/time_aggregate_dataset.ft')
time_aggregated_dataset = time_aggregated_dataset[~time_aggregated_dataset.treatment.isin(dynamic_treatments)]
time_aggregated_dataset = time_aggregated_dataset.query('round <= 5 and time <= 120')

## Keep relevant columns and prepare train-test loader

In [4]:
key_columns = ['treatment', 'game', 'round', 'time', 'n_unique_deals_round']
feature_cols = [col for col in time_aggregated_dataset.columns.values if "change" not in col and "running_" in col] #+ ['realized_price']# + ['n_deal_prices_round'] + ['round']
rounds = range(1,5)
n_deal_prices = range(0,6)
pdl = PandasDataLoader(sample_splits_df, time_aggregated_dataset)
target_col = 'ce_round'

## Fit and evaluate models

In [5]:
np.random.seed(1)
all_results = []
regression_res = []

for i in tqdm(range(pdl.max_samples)):
    train_df, test_df = pdl.get_sample_split_dataset(i)
    for rd in rounds:
        train_query = 'round <= ' + str(rd) # model performance seems to drop if we include all the dataset  in small validation sets.
        formula = target_col+'~ ('+'+'.join(feature_cols)+')-1' # including feedback setting as feedbackse∈g:featcols...feedbacksetting:(featcols... degrades performance seems ot degrade 1% in validation set.
        model = smf.rlm(formula, data=train_df.query(train_query))
        best_model = model.fit()

        a = (best_model.summary2().tables[1]).stack().to_frame().T.swaplevel(-2, -1, axis=1)
        a['sample_id'] = i
        a['round'] = rd
        regression_res.append(a)
        
        for n_deal_price in n_deal_prices:
            key = (rd, n_deal_price)
            test_query = 'round == ' + str(rd) + ' and n_unique_deals_round ==  ' + str(n_deal_price)
            sub_test_set = test_df.query(test_query)            
            prediction = best_model.predict(sub_test_set)            
            test_targets = sub_test_set[target_col]

            result_test_df = sub_test_set[key_columns].copy()
            result_test_df.loc[:, 'ce_ape'] = (np.abs(prediction - test_targets)/test_targets)
            result_test_df.loc[:, 'sample_id'] = i

            all_results.append(result_test_df)

  0%|          | 0/50 [00:00<?, ?it/s]

## Combine Results

In [6]:
all_results_df = pd.concat(all_results, ignore_index = True)
regression_data_df = pd.concat(regression_res, axis=0, ignore_index = True)
all_results_df['model'] = model_name

## Persist Results

In [7]:
all_results_df.to_feather('../../../data/results/ce_price/'+model_name+'.ft')
regression_data_df.reset_index().to_feather('../../../data/results/ce_price/'+model_name+'_data.ft')